# BOAZ BASE 2주차 과제 — RNN · LSTM · GRU

이 노트북을 **본인 구글 드라이브로 사본 저장**한 뒤 진행하고, 완성한 `.ipynb`를 깃허브에 업로드해 주세요.

**구성**
1. RNN 튜토리얼
2. RNN 과제

튜토리얼은 **완성된 코드**로 제공됩니다.
먼저 셀을 순서대로 실행하면서 출력과 주석을 천천히 읽어보면서 진행하는걸 추천합니다.

---
## 1. RNN 튜토리얼

**Sequential data**를 다루는 가장 기본 모델이 RNN입니다.
이전 시점의 hidden state를 다음 시점으로 넘기면서 **과거 정보를 계승**합니다.

**튜토리얼**: 길이 12짜리 0/1 시퀀스를 입력받아, 그 안에 `1-0-1` 패턴이 한 번이라도 나오면 1, 아니면 0을 맞히는 이진 분류기입니다.

구조는 세 가중치(입력→은닉 $W_{xh}$, 은닉→은닉 $W_{hh}$, 은닉→출력 $W_{hy}$)와 $\tanh$ 활성화가 `nn.RNN` 안에 그대로 들어 있다고 보면 됩니다.

In [1]:
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from itertools import product

SEED = 0
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
# 정답 label을 만드는 함수
# seq 안에 연속된 3칸이 1-0-1 이면 1, 아니면 0
def has_101_pattern(seq):
    for i in range(len(seq) - 2):
        if seq[i] == 1 and seq[i+1] == 0 and seq[i+2] == 1:
            return 1
    return 0

In [3]:
# PyTorch에서 Dataset은 "데이터 1개를 어떻게 꺼낼지"를 표준화한 클래스입니다.
# 이걸 상속받아 내 task에 맞는 커스텀 Dataset을 만들어 모델에 기입합니다.
class PatternDataset(Dataset):
    def __init__(self, seqs):
        self.data = [(s, has_101_pattern(s)) for s in seqs]

    def __len__(self):
        return len(self.data)
        # 전체 sample 개수

    def __getitem__(self, idx):
        seq, label = self.data[idx]
        x = torch.tensor(seq, dtype=torch.long)
        # (T,)  embedding은 정수 인덱스를 받으므로 long
        y = torch.tensor([label], dtype=torch.float)
        # (1,)  BCEWithLogitsLoss가 float label을 기대
        return x, y

# 데이터 생성 및 로더 세팅
all_seqs = [list(s) for s in product([0, 1], repeat=12)]
pos = [s for s in all_seqs if has_101_pattern(s) == 1]
neg = [s for s in all_seqs if has_101_pattern(s) == 0]

rng = random.Random(SEED)
rng.shuffle(pos)
rng.shuffle(neg)
n = min(len(pos), len(neg))
balanced = pos[:n] + neg[:n]
rng.shuffle(balanced)

n_train = int(len(balanced) * 0.8)
train_seqs, val_seqs = balanced[:n_train], balanced[n_train:]

train_loader = DataLoader(PatternDataset(train_seqs), batch_size=64, shuffle=True)
# train은 섞어서 안정적으로
val_loader   = DataLoader(PatternDataset(val_seqs), batch_size=256, shuffle=False)
# val은 섞을 필요 없음

print(f"전체 {len(all_seqs)}개 중 label=1 : {len(pos)} ({len(pos)/len(all_seqs):.1%})")
print(f"train {len(train_seqs)} / val {len(val_seqs)}")
print("겹치는 시퀀스 :", len(set(map(tuple, train_seqs)) & set(map(tuple, val_seqs))), "개")

전체 4096개 중 label=1 : 3015 (73.6%)
train 1729 / val 433
겹치는 시퀀스 : 0 개


**RNN 모델.** PyTorch 모델은 보통 `nn.Module`을 상속해서 만듭니다. `embed → rnn → fc` 순서로 통과시켜 마지막 hidden으로 분류합니다.

In [4]:
class SimpleRNNClassifier(nn.Module):
    def __init__(self, vocab_size=2, embed_dim=8, hidden_dim=16):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        # (0/1) 토큰 -> 벡터

        self.rnn = nn.RNN(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)
        # 시퀀스를 왼쪽부터 읽으며 hidden 갱신

        self.fc = nn.Linear(hidden_dim, 1)
        # 마지막 hidden -> 이진 분류 점수(logit)

    def forward(self, x):
        # x: (B, T) 0/1 토큰,  B=batch, T=시퀀스 길이
        emb = self.embed(x)
        # (B, T, E)
        out, h_n = self.rnn(emb)
        # out: (B, T, H) 모든 시점 hidden / h_n: (1, B, H) 마지막 hidden
        last_h = h_n[-1]
        # (B, H)
        logit = self.fc(last_h)
        # (B, 1)
        return logit, out, h_n

    def forward_with_trace(self, x):
        # 시간에 따른 hidden 변화를 눈으로 보기 위한 함수 (x는 (1, T) 단일 시퀀스 권장)
        emb = self.embed(x)
        # (1, T, E)
        out, h_n = self.rnn(emb)
        # out: (1, T, H)
        logit = self.fc(h_n[-1])
        # (1, 1)
        return logit, out.squeeze(0)
        # logit:(1,1), trace:(T, H)

학습(train) → 검증(val) → 마지막에 demo 시퀀스로 예측 확률과 hidden 변화를 출력합니다.

In [5]:
def train_rnn():
    model = SimpleRNNClassifier(embed_dim=8, hidden_dim=16).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

    for epoch in range(1, 6):
        model.train()
        total_loss = 0.0

        # 1. 학습 루프
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logit, _, _ = model(x)
            loss = criterion(logit, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)

        train_loss = total_loss / len(train_seqs)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():

            # 2. 검증 루프
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                prob = torch.sigmoid(model(x)[0])
                pred = (prob >= 0.5).float()
                correct += (pred == y).sum().item()
                total += y.numel()

        # 3. 에포크 결과 출력
        print(f"[Epoch {epoch}] train_loss={train_loss:.4f}  val_acc={correct/total:.4f}")

    return model

# ---- DEMO: hidden state ----
def demo(model, demo_seq):
    model.eval()
    x_demo = torch.tensor([demo_seq], dtype=torch.long).to(device)
    # (1, T)

    with torch.no_grad():
        logit, trace = model.forward_with_trace(x_demo)
        prob = torch.sigmoid(logit).item()

        print("\n=== DEMO ===")
        print("demo_seq       :", demo_seq)
        print("정답(has 101)  :", has_101_pattern(demo_seq))
        print("예측 확률(P=101):", round(prob, 4))

        trace = trace.cpu()
        # (T, H)
        print("\nhidden state 추적 (앞 3스텝 + 뒤 3스텝, 앞 6차원만):")
        T = len(demo_seq)
        for t in list(range(3)) + list(range(T-3, T)):
            vec = [round(v, 3) for v in trace[t][:6].tolist()]
            print(f"  t={t:2d}, x_t={demo_seq[t]} -> h_t[:6]={vec}")

In [6]:
rnn_model = train_rnn()

demo_has = [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
demo(rnn_model, demo_has)

demo_no  = [1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0]
demo(rnn_model, demo_no)

[Epoch 1] train_loss=0.6571  val_acc=0.5381
[Epoch 2] train_loss=0.6594  val_acc=0.6836
[Epoch 3] train_loss=0.5738  val_acc=0.7691
[Epoch 4] train_loss=0.4141  val_acc=0.9400
[Epoch 5] train_loss=0.1163  val_acc=1.0000

=== DEMO ===
demo_seq       : [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
정답(has 101)  : 1
예측 확률(P=101): 0.9895

hidden state 추적 (앞 3스텝 + 뒤 3스텝, 앞 6차원만):
  t= 0, x_t=1 -> h_t[:6]=[-0.577, -0.849, 0.662, 0.989, -0.904, -0.609]
  t= 1, x_t=0 -> h_t[:6]=[-0.492, 0.814, 0.875, 0.984, 0.932, 0.966]
  t= 2, x_t=1 -> h_t[:6]=[-0.71, -0.908, 0.594, 0.952, 0.76, -0.851]
  t= 9, x_t=0 -> h_t[:6]=[0.924, -0.96, 0.883, 0.753, 0.928, -0.982]
  t=10, x_t=0 -> h_t[:6]=[0.924, -0.959, 0.882, 0.75, 0.929, -0.982]
  t=11, x_t=0 -> h_t[:6]=[0.924, -0.959, 0.881, 0.748, 0.928, -0.981]

=== DEMO ===
demo_seq       : [1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0]
정답(has 101)  : 0
예측 확률(P=101): 0.0964

hidden state 추적 (앞 3스텝 + 뒤 3스텝, 앞 6차원만):
  t= 0, x_t=1 -> h_t[:6]=[-0.577, -0.849, 0.662, 0.989, -0.904, -0

### Q1. 위 코드의 출력을 분석하고, `demo_has`와 `demo_no` 두 입력을 넣었을 때 예측 확률·hidden state가 어떻게 달랐는지 비교·서술하세요.

Ans)
`demo_has`는 입력 시퀀스에 `101` 패턴이 포함되어 있는 데이터이며, 모델은 예측 확률 0.9895(약 98.95%)를 출력하여 `101` 패턴이 존재한다고 판단하였다. 반면 `demo_no`는 `101` 패턴이 없는 데이터로, 예측 확률 0.0964(약 9.64%)를 출력하여 `101` 패턴이 없다고 예측하였다.

Hidden state를 비교해 보면, 두 입력 모두 첫 번째 토큰에서는 동일한 hidden state에서 시작하지만, 이후 입력 토큰의 차이에 따라 hidden state가 점차 다른 방향으로 변화하였다. `demo_has`에서는 `101` 패턴을 읽은 이후 hidden state가 특정 값으로 수렴하며 마지막까지 비교적 안정적인 상태를 유지하였다. 특히 마지막 몇 단계(t=9~11)의 hidden state 값이 거의 일정하게 유지되어, RNN이 `101` 패턴을 기억한 채 최종 예측에 활용하고 있음을 확인할 수 있었다.

반면 `demo_no`에서는 `101` 패턴이 등장하지 않아 hidden state가 입력 토큰에 따라 계속 변화하였으며, 마지막 단계에서도 `demo_has`와는 다른 값으로 형성되었다. 이러한 hidden state의 차이가 최종 출력(logit)에 반영되어 낮은 예측 확률(0.0964)이 계산되었다.

즉, RNN은 입력 시퀀스를 순차적으로 처리하면서 hidden state에 이전 정보를 누적 저장하고, `101` 패턴이 존재하는 경우와 존재하지 않는 경우 서로 다른 hidden state를 형성한다. 이 hidden state의 차이가 최종 예측 확률의 차이로 이어지는 것을 확인할 수 있었다.


###Q2. 아래 코드의 `TODO` 빈칸(`____`)을 모두 채운 뒤 순서대로 실행하고, 맨 아래 서술형 질문에 답하면 됩니다.

In [1]:
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
# 정답 label을 만드는 함수
def has_101_pattern(seq):
    for i in range(len(seq) - 2):
        if seq[i] == 1 and seq[i+1] == 0 and seq[i+2] == 1:
            return 1
    return 0

# 학습용 커스텀 Dataset
class PatternDataset(Dataset):
    def __init__(self, n_samples=5000, seq_len=12):
        self.data = []
        for _ in range(n_samples):
            seq = [random.randint(0, 1) for _ in range(seq_len)]
            self.data.append((seq, has_101_pattern(seq)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq, label = self.data[idx]
        x = torch.tensor(seq, dtype=torch.long)
        # (T,)  embedding은 정수 인덱스를 받으므로 long
        y = torch.tensor([label], dtype=torch.float)
        # (1,)  BCEWithLogitsLoss는 float label을 기대
        return x, y

### 1) RNN 모델 빈칸 채우기
`embed → rnn → fc` 순서로 시퀀스를 통과시켜, *마지막 hidden state*로 분류 점수(logit) 출력

힌트: `nn.Embedding(vocab_size, embed_dim)`, `nn.RNN(input_size=?, hidden_size=?, batch_first=True)`, `out, h_n = self.rnn(emb)`, 마지막 hidden은 `h_n[-1]`.

In [3]:
class SimpleRNNClassifier(nn.Module):
    def __init__(self, vocab_size=2, embed_dim=8, hidden_dim=16):
        super().__init__()
        # TODO: (0/1) 토큰을 벡터로 바꾸는 임베딩 레이어
        self.embed = nn.Embedding(vocab_size, embed_dim)
        # TODO: RNN 레이어 (input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)
        self.rnn = nn.RNN(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)
        # 마지막 hidden -> 이진 분류 점수(logit) 1개
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x: (B, T) 0/1 토큰   (B=batch, T=시퀀스 길이)
        # TODO: 임베딩 통과 -> (B, T, E)
        emb = self.embed(x)
        # TODO: RNN에 넣어 out, h_n 받기   out:(B,T,H) / h_n:(1,B,H)
        out, h_n = self.rnn(emb)
        # TODO: 마지막 시점의 hidden state 얻기 -> (B, H)
        last_h = h_n[-1]
        logit = self.fc(last_h)
        # (B, 1)
        return logit

###2) 학습 루프 빈칸 채우기
학습(train) → 검증(val)을 5 epoch 반복한다. loss는 `BCEWithLogitsLoss`, optimizer는 `Adam`을 씁니다.

In [6]:
def train():
    train_ds = PatternDataset(n_samples=6000, seq_len=12)
    val_ds   = PatternDataset(n_samples=1000, seq_len=12)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    # train은 섞어서
    val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False)
    # val은 안 섞음

    model = SimpleRNNClassifier(embed_dim=8, hidden_dim=16).to(device)
    # TODO: loss 함수 (BCEWithLogitsLoss)
    criterion = nn.BCEWithLogitsLoss()
    # TODO: optimizer (Adam, lr=1e-3)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(1, 6):
        model.train()
        total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logit = model(x)

            # TODO: 이전 gradient 초기화
            optimizer.zero_grad()
            # TODO: loss 계산 (logit, y)
            loss = criterion(logit, y)
            # TODO: 역전파
            loss.backward()
            # TODO: optimizer 한 스텝
            optimizer.step()


            total_loss += loss.item() * x.size(0)
        train_loss = total_loss / len(train_ds)

        # ---- 검증 ----
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                # TODO: logit -> 확률 (sigmoid)
                prob = torch.sigmoid(model(x)[0])
                # TODO: 0.5 이상이면 1로 예측 (float)
                pred = (prob >= 0.5).float()
                correct += (pred == y).sum().item()
                total += y.numel()
        print(f"[Epoch {epoch}] train_loss={train_loss:.4f}  val_acc={correct/total:.4f}")

    return model

model = train()

[Epoch 1] train_loss=0.5792  val_acc=0.7340
[Epoch 2] train_loss=0.5173  val_acc=0.7340
[Epoch 3] train_loss=0.4791  val_acc=0.7340
[Epoch 4] train_loss=0.3781  val_acc=0.7340
[Epoch 5] train_loss=0.2100  val_acc=0.7340


###3) 직접 넣어보기 (제공)
학습된 모델에 아래 두 시퀀스를 넣어 예측 확률을 확인하세요.

In [7]:
def predict(model, seq):
    model.eval()
    x = torch.tensor([seq], dtype=torch.long).to(device)
    with torch.no_grad():
        prob = torch.sigmoid(model(x)).item()
    print(f"seq={seq}  정답(has 101)={has_101_pattern(seq)}  예측확률(P=101)={prob:.4f}")

predict(model, [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0])
# 1-0-1 있음 -> 높아야 함

predict(model, [1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0])
# 1-0-1 없음 -> 낮아야 함

seq=[1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]  정답(has 101)=1  예측확률(P=101)=0.9349
seq=[1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0]  정답(has 101)=0  예측확률(P=101)=0.4698


### Q3. 두 입력의 예측 확률을 비교하고, RNN이 이 문제를 어떻게 푸는지 서술하세요.
(hidden state가 시퀀스를 왼쪽부터 읽으며 과거 정보를 어떻게 전달하는지 언급하면 좋습니다.)

Ans) 첫 번째의 예측 확률은 0.9349로 매우 높게 나타나 `101` 패턴이 존재한다고 판단하였다. 반면 두 번째의 예측 확률은 0.4698로 매우 낮아 `101` 패턴이 존재하지 않는다고 판단하였다. 즉, 두 입력은 확률 차이가 크게 나타났으며, 모델이 두 시퀀스를 명확하게 구분하고 있음을 확인할 수 있다.

RNN은 입력 시퀀스를 **왼쪽부터 한 글자씩 순차적으로 읽으며**, 각 시점에서 이전 hidden state와 현재 입력을 함께 이용하여 새로운 hidden state를 생성한다. hidden state에는 지금까지 읽은 입력 정보가 계속 누적되어 전달된다. `첫 번째 시퀀스`에서는 입력을 읽는 과정에서 `101` 패턴이 등장하자 hidden state에 해당 정보가 저장되고, 이후 마지막 시점까지 그 정보가 유지되어 높은 예측 확률(0.9349)을 출력하였다. 반면 `두 번재 시퀀스`에서는 `101` 패턴이 끝까지 나타나지 않아 hidden state에 해당 정보를 저장하지 못했고, 최종적으로 낮은 예측 확률(0.4698)을 출력하였다.

RNN은 hidden state를 통해 과거 정보를 다음 시점으로 전달하면서 시퀀스 전체의 문맥을 기억하고, 마지막 hidden state를 이용하여 `101` 패턴의 존재 여부를 판단하는 방식으로 이 문제를 해결한다.
